In [1]:
import torch
import random
import torch.nn as nn


In [2]:
class SimpleEnviroment:
    def __init__(self):
        self.states = ["s1", "s2"]
        self.actions= ["a1", "a2", "a3"]

        self.trans = {
            ("s1", "a1"): {("s1", +1): 1 },
            ("s1", "a2"): {("s1", +4): 0.7, ("s2", +5) : 0.3 },
            ("s2", "a1"): {("s2", +1): 1 },
            ("s2", "a2"): {("s1", -3): 0.8, ("s2", +2) : 0.2 },
            ("s2", "a3"): {("s2", 0): 1 }
        }

    def reset(self):
        self.current_state = random.choice(self.states)
        return self.current_state 
    
    def step(self, action):
        trans_value = self.trans[(self.current_state, action)]
        next_state, reward = random.choices(list(trans_value.keys()), weights=list(trans_value.values()))[0]
        self.current_state = next_state
        return next_state, reward
    


In [3]:

class GridEnvironment:
    def __init__(self, size):
        self.size = size
        # self.terminate_state = [(0,0), (size-1, size-1)]
        self.terminate_state = [(size-1, size-1)]
        self.states = [(i,j) for i in range(self.size) for j in range(self.size)
                       if (i,j) not in self.terminate_state]
        self.actions = [(1,0), (0,1), (-1,0), (0,-1)]

    def reset(self): 
        self.current_state = random.choice(self.states)
        return self.current_state
    
    def step(self, action):
        next_state = (self.current_state[0] + action[0], self.current_state[1] + action[1])
        # Check if next_state is valid (within grid boundaries)
        if not (0 <= next_state[0] < self.size and 0 <= next_state[1] < self.size):
            # If out of bounds, stay in the current state
            next_state = self.current_state
        reward = -1 
        self.current_state = next_state
        return next_state, reward

def deterministic_policy(env, state):
    valid_actions = []
    for action in env.actions:
        next_state = (state[0] + action[0], state[1] + action[1])
        if 0 <= next_state[0] < env.size and 0 <= next_state[1] < env.size:
            valid_actions.append(action)
    
    if valid_actions:
        return random.choice(valid_actions)
    return (0, 0)  # Default action if no valid actions (shouldn't happen with boundary checking)

def run_episode(env, max_steps=50):
    state = env.reset()
    total_sarsa = []
    
    for _ in range(max_steps):
        action = deterministic_policy(env, state)
        next_state, reward = env.step(action)
        
        if next_state in env.terminate_state:
            print("Reached terminal state:", next_state)
            total_sarsa.extend([state, action, reward, next_state, "end"])
            break
        else:
            total_sarsa.extend([state, action, reward])
        
        state = next_state
    
    return total_sarsa

In [4]:
def run_episode_update_q(env, max_steps=100):
    state = env.reset()
    total_sarsa = []
    
    for _ in range(max_steps):
        action = deterministic_policy(env, state)
        next_state, reward = env.step(action)
        
        if next_state in env.terminate_state:
            print("Reached terminal state:", next_state)
            total_sarsa.extend([state, action, reward, next_state, 100])
            break
        else:
            total_sarsa.extend([state, action, reward])
        
        state = next_state
    
    return total_sarsa

In [5]:
class QFunction(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(QFunction, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qfunction=QFunction(2, 30, 4)
qfunction = qfunction.to(device)

In [7]:
device_test = next(qfunction.parameters()).device

In [8]:
device_test

device(type='cuda', index=0)

In [9]:
gamma= 0.5
theta = 1e-10


In [10]:
env=GridEnvironment(6)



In [11]:
state = env.reset()
episodes = []
for i in range(0, 2000):
    episode=run_episode_update_q(env, 1000)
    episodes.append(episode)

Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached terminal state: (5, 5)
Reached 

In [12]:

optimizer = torch.optim.Adam(
    qfunction.parameters(),
    lr=0.001,           # Learning rate
    betas=(0.9, 0.999), # Exponential decay rates for moment estimates
    eps=1e-8,           # Term added for numerical stability
    weight_decay=0      # L2 penalty (regularization)
)

# Alternative optimizers
# sgd_optimizer = torch.optim.SGD(qfunction.parameters(), lr=0.01, momentum=0.9)
# rmsprop_optimizer = torch.optim.RMSprop(qfunction.parameters(), lr=0.01)


In [13]:
def update_q(episodes, qfunction, optimizer, gamma=0.5):
    device = next(qfunction.parameters()).device
    action_to_idx = {(1,0): 0, (0,1): 1, (-1,0): 2, (0,-1): 3}  # Map actions to indices
    
    for k in range(0, 20):
        print("k update ->", k)
        for q in range(0, len(episodes)):
            for i in range(0, len(episodes[q])-5, 3):
                state = episodes[q][i]
                action = episodes[q][i+1]
                reward = episodes[q][i+2]
                next_state = episodes[q][i+3]
                next_action = episodes[q][i+4]
                
                # Convert state to tensor (only state, not action)
                state_tensor = torch.FloatTensor(list(state)).to(device)
                next_state_tensor = torch.FloatTensor(list(next_state)).to(device)
                
                # Get Q-values for all actions
                q_values = qfunction(state_tensor)
                
                # Get action index
                action_idx = action_to_idx[action]
                
                with torch.no_grad():
                    # Get next state Q-values
                    next_q_values = qfunction(next_state_tensor)
                    # Get Q-value for next action
                    next_action_idx = action_to_idx[next_action]
                    next_q_value = next_q_values[next_action_idx]
                    # Calculate target
                    target = reward + gamma * next_q_value.item()
                
                # Create target tensor with same shape as q_values
                target_q_values = q_values.clone().detach()
                # Update only the specific action's Q-value
                target_q_values[action_idx] = target
                
                # Compute loss
                loss = nn.MSELoss()(q_values, target_q_values)
                
                # Update network
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Handle terminal state
            try:
                state = episodes[q][-5]
                action = episodes[q][-4]
                reward = episodes[q][-1]
                
                state_tensor = torch.FloatTensor(list(state)).to(device)
                q_values = qfunction(state_tensor)
                
                action_idx = action_to_idx[action]
                
                # Create target tensor
                target_q_values = q_values.clone().detach()
                target_q_values[action_idx] = reward
                
                loss = nn.MSELoss()(q_values, target_q_values)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            except Exception as e:
                print(e)
                print(f"State: {state}, Action: {action}")
        
    return qfunction

        

In [14]:
final_qfucntion=update_q(episodes,qfunction,optimizer )

k update -> 0
k update -> 1
k update -> 2
k update -> 3
k update -> 4
k update -> 5
k update -> 6
k update -> 7
k update -> 8
k update -> 9
k update -> 10
k update -> 11
k update -> 12
k update -> 13
k update -> 14
k update -> 15
k update -> 16
k update -> 17
k update -> 18
k update -> 19


In [15]:
final_qfucntion =final_qfucntion.to(device)
device_test = next(final_qfucntion.parameters()).device

In [16]:
device_test

device(type='cuda', index=0)

In [33]:
final=[0,0] 
state_tensor = torch.FloatTensor(final).to(device)
output_state=final_qfucntion(state_tensor)
print(output_state)

tensor([-1.8902, -2.3048, -1.9990, -1.8775], device='cuda:0',
       grad_fn=<ViewBackward0>)


In [ ]:
action_to_idx = {(1,0): 0, (0,1): 1, (-1,0): 2, (0,-1): 3}

In [36]:
def follow_optimal_policy(final_qfunction, start_state=(0,1), goal_state=(5,5), grid_size=6):
    device = next(final_qfunction.parameters()).device
    idx_to_action = {0: (1,0), 1: (0,1), 2: (-1,0), 3: (0,-1)}
    
    path = [start_state]
    current_state = start_state
    
    # Prevent infinite loops
    max_steps = 100
    steps = 0
    
    while current_state != goal_state and steps < max_steps:
        # Convert state to tensor
        state_tensor = torch.FloatTensor(list(current_state)).to(device)
        
        # Get Q-values for all actions
        with torch.no_grad():
            q_values = final_qfunction(state_tensor)
        
        # Find action with highest Q-value
        best_action_idx = torch.argmax(q_values).item()
        best_action = idx_to_action[best_action_idx]
        
        # Calculate next state
        next_state = (current_state[0] + best_action[0], current_state[1] + best_action[1])
        
        # Check if next_state is valid (within grid boundaries)
        if not (0 <= next_state[0] < grid_size and 0 <= next_state[1] < grid_size):
            # If out of bounds, stay in the current state
            next_state = current_state
        
        # Add to path and update current state
        path.append([next_state, q_values])
        current_state = next_state
        steps += 1
    
    return path



In [37]:
follow_optimal_policy(final_qfucntion)

[(0, 1),
 [(1, 1), tensor([-1.8367, -2.1740, -1.9870, -1.8842], device='cuda:0')],
 [(2, 1), tensor([-1.7855, -2.0487, -1.9755, -1.8907], device='cuda:0')],
 [(3, 1), tensor([-1.7342, -1.9233, -1.9640, -1.8972], device='cuda:0')],
 [(4, 1), tensor([-1.6830, -1.7980, -1.9524, -1.9036], device='cuda:0')],
 [(5, 1), tensor([-1.6244, -1.6547, -1.9393, -1.9110], device='cuda:0')],
 [(5, 2), tensor([-1.5102, -1.3753, -1.9136, -1.9254], device='cuda:0')],
 [(5, 3), tensor([ 0.5183,  0.6895, -1.8172, -1.8577], device='cuda:0')],
 [(5, 4), tensor([13.9827, 14.4812, -1.1669, -1.4180], device='cuda:0')],
 [(5, 5), tensor([85.4688, 99.5782,  1.7619,  0.7179], device='cuda:0')]]